# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ameen740/Internship_Flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_Token")

con = duckdb.connect()

rel = "hf://datasets/FlyRank/internship-warehouse"

con.execute(
    f"CREATE OR REPLACE SECRET hf_token "
    f"(TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')"
)

test = con.sql(f"""
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 5
""")

print("✅ FlyRank warehouse connection successful!")
print("✅ March 2026 data is accessible.")
print(test)

✅ FlyRank warehouse connection successful!
✅ March 2026 data is accessible.
┌─────────────┬─────────────────────────┬──────────────────────────┬────────────────┬────────────────┬────────────────────┬────────────────────┬─────────────────┬────────────┬──────────────────┬───────────────────┬───────────────┬──────────────┬───────────┬──────────────────────┬──────────────────────────┬──────────────────┬─────────────────┬───────────────────┬─────────────────┬───────────────┬─────────────┬────────────┬───────────────┬───────────┬────────────┬───────────┬─────────┬──────────┬───────────────┬─────────┐
│ report_date │     client_hash_id      │     content_hash_id      │ client_has_gsc │ client_has_ga4 │ gsc_data_available │ ga4_data_available │ gsc_impressions │ gsc_clicks │ gsc_sum_position │ gsc_avg_position  │ ga4_pageviews │ ga4_sessions │ ga4_users │ ga4_engaged_sessions │ ga4_total_engagement_sec │ sessions_organic │ sessions_direct │ sessions_referral │ sessions_social │ sessions_paid │

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My baseline rule will prioritize content items that have search visibility but a possible CTR improvement opportunity. The rule will use historical Google Search Console signals that are available at the decision moment. Items with enough impressions and relatively low CTR will receive higher priority for review. The rule supports a content review decision and does not automatically change the content.

Reason codes:

- CTR_FIX_CANDIDATE — the content has enough search impressions and may have an opportunity to improve CTR.
- NO_ACTION — the available signals do not provide enough evidence for a CTR review.

Signal 1 verdict: CONFIRMED

Signal 2 verdict: CONFIRMED

The March 2026 bucket checks support both signals. Higher-impression content shows a lower average CTR, while CTR decreases as search position gets worse. These are observed directional patterns, not causal proof.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Question 1: Check two signals used by the baseline rule
# Signal 1 = search volume (impressions)
# Signal 2 = CTR, with search position as context
import pandas as pd

signal_data = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    gsc_data_available
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
  AND gsc_impressions > 0
""").df()

# Calculate CTR as a percentage
signal_data["ctr_pct"] = (
    100.0 * signal_data["gsc_clicks"] / signal_data["gsc_impressions"]
)

# ---------------------------------------------------------
# SIGNAL 1: Search volume / impressions
# ---------------------------------------------------------

signal_data["impression_bucket"] = pd.cut(
    signal_data["gsc_impressions"],
    bins=[0, 100, 1000, float("inf")],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)

impression_check = (
    signal_data
    .groupby("impression_bucket", observed=False)
    .agg(
        n=("gsc_impressions", "size"),
        avg_impressions=("gsc_impressions", "mean"),
        avg_ctr_pct=("ctr_pct", "mean")
    )
    .reset_index()
)

print("SIGNAL 1 — SEARCH VOLUME / IMPRESSIONS")
display(impression_check)

# ---------------------------------------------------------
# SIGNAL 2: CTR vs search position
# ---------------------------------------------------------

position_data = signal_data[
    signal_data["gsc_avg_position"] > 0
].copy()

position_data["position_bucket"] = pd.cut(
    position_data["gsc_avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["Top 3", "4-10", "11-20", "21+"],
    include_lowest=True
)

position_check = (
    position_data
    .groupby("position_bucket", observed=False)
    .agg(
        n=("gsc_avg_position", "size"),
        avg_position=("gsc_avg_position", "mean"),
        avg_ctr_pct=("ctr_pct", "mean")
    )
    .reset_index()
)

print("\nSIGNAL 2 — CTR VS SEARCH POSITION")
display(position_check)

# ---------------------------------------------------------
# Verdicts
# ---------------------------------------------------------

print("\nSIGNAL VERDICTS")
print("Signal 1 — Search volume: CONFIRMED if higher-volume buckets show useful CTR opportunity.")
print("Signal 2 — CTR vs position: CONFIRMED if CTR changes meaningfully across position buckets.")
print("The actual bucket tables above are the evidence for the verdicts.")

SIGNAL 1 — SEARCH VOLUME / IMPRESSIONS


,impression_bucket,n,avg_impressions,avg_ctr_pct
0,Low,2977578,20.379317,0.308679
1,Medium,601123,268.091439,0.307052
2,High,32360,1817.696323,0.271525



SIGNAL 2 — CTR VS SEARCH POSITION


,position_bucket,n,avg_position,avg_ctr_pct
0,Top 3,564173,1.807196,0.491821
1,4-10,1456122,6.059994,0.347264
2,11-20,519223,14.330876,0.276991
3,21+,908354,43.888638,0.128915



SIGNAL VERDICTS
Signal 1 — Search volume: CONFIRMED if higher-volume buckets show useful CTR opportunity.
Signal 2 — CTR vs position: CONFIRMED if CTR changes meaningfully across position buckets.
The actual bucket tables above are the evidence for the verdicts.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# Load March 2026 data
df = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_data_available
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
WHERE gsc_data_available IS TRUE
  AND gsc_impressions > 0
""").df()

# Calculate CTR
df["ctr_pct"] = (
    100.0 * df["gsc_clicks"] / df["gsc_impressions"]
)

# Calculate the baseline priority score
df["score"] = (
    df["gsc_impressions"] / (df["ctr_pct"] + 0.1)
)

# Default reason and action
df["reason_code"] = "NO_ACTION"
df["action"] = "NO_ACTION"

# Identify CTR review candidates
candidate = (
    (df["gsc_impressions"] >= 100) &
    (df["ctr_pct"] < 2.0)
)

df.loc[candidate, "reason_code"] = "CTR_FIX_CANDIDATE"
df.loc[candidate, "action"] = "REVIEW_CTR"

# Rank all rows by score
df = df.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

df["rank"] = df.index + 1

# Create the final ranked queue
queue = df[
    [
        "rank",
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr_pct",
        "score",
        "reason_code",
        "action"
    ]
]

# Create output folder
os.makedirs("work/outputs", exist_ok=True)

# Write the required CSV
output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

# Confirm the result
print("Baseline ranked queue created successfully.")
print("Number of rows:", len(queue))
print("Output file:", output_path)

# Display the top 20
queue.head(20)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Baseline ranked queue created successfully.
Number of rows: 3611061
Output file: work/outputs/baseline_action_score.csv


,rank,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr_pct,score,reason_code,action
0,1,2026-03-28,client_23a62021009f63c4,content_44f34c0a90047651,40084,1,0.002495,391083.403758,CTR_FIX_CANDIDATE,REVIEW_CTR
1,2,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,37368,0,0.000000,373680.000000,CTR_FIX_CANDIDATE,REVIEW_CTR
2,3,2026-03-04,client_62f4a7e64f5e0096,content_34a70fea29d15f24,39003,2,0.005128,371005.538375,CTR_FIX_CANDIDATE,REVIEW_CTR
3,4,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,33383,0,0.000000,333830.000000,CTR_FIX_CANDIDATE,REVIEW_CTR
4,5,2026-03-27,client_23a62021009f63c4,content_44f34c0a90047651,32958,0,0.000000,329580.000000,CTR_FIX_CANDIDATE,REVIEW_CTR
5,6,2026-03-31,client_73cda7b4e4f265ea,content_fec55986a1868d62,31472,0,0.000000,314720.000000,CTR_FIX_CANDIDATE,REVIEW_CTR
6,7,2026-03-29,client_23a62021009f63c4,content_44f34c0a90047651,32756,2,0.006106,308710.880424,CTR_FIX_CANDIDATE,REVIEW_CTR
7,8,2026-03-25,client_23a62021009f63c4,content_44f34c0a90047651,30964,1,0.003230,299952.851958,CTR_FIX_CANDIDATE,REVIEW_CTR
8,9,2026-03-02,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,28973,0,0.000000,289730.000000,CTR_FIX_CANDIDATE,REVIEW_CTR
9,10,2026-03-01,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,28947,0,0.000000,289470.000000,CTR_FIX_CANDIDATE,REVIEW_CTR


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

I reviewed the top 20 items produced by the baseline rule. The action and reason code come directly from the rule. The confidence note reflects how much search evidence is available, while the "what would make it wrong" note identifies conditions that could make the recommendation unreliable. These are decision-support recommendations and should be reviewed by a content specialist before action.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Question 3: Review the top 20 ranked items

top20 = queue.head(20).copy()

# Add a confidence note based on search impressions
def confidence_note(impressions):
    if impressions >= 1000:
        return "Higher confidence: strong search volume."
    elif impressions >= 300:
        return "Moderate confidence: useful search volume."
    else:
        return "Lower confidence: limited search volume."

top20["confidence_note"] = top20["gsc_impressions"].apply(
    confidence_note
)

# Add a note explaining what could make the recommendation wrong
def wrong_reason(row):
    if row["gsc_impressions"] < 300:
        return "Could be wrong because search volume is limited."
    elif row["gsc_clicks"] == 0:
        return "Could be wrong because the item has no observed clicks."
    elif row["ctr_pct"] >= 2.0:
        return "Could be wrong because CTR is not especially low."
    else:
        return "Could be wrong if the observed CTR does not reflect the true improvement opportunity."

top20["what_would_make_it_wrong"] = top20.apply(
    wrong_reason,
    axis=1
)

# Show the required review information
top20_review = top20[
    [
        "rank",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

top20_review

,rank,content_hash_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_44f34c0a90047651,REVIEW_CTR,CTR_FIX_CANDIDATE,Higher confidence: strong search volume.,Could be wrong if the observed CTR does not re...
1,2,content_945d6ff91386c817,REVIEW_CTR,CTR_FIX_CANDIDATE,Higher confidence: strong search volume.,Could be wrong because the item has no observe...
2,3,content_34a70fea29d15f24,REVIEW_CTR,CTR_FIX_CANDIDATE,Higher confidence: strong search volume.,Could be wrong if the observed CTR does not re...
3,4,content_fec55986a1868d62,REVIEW_CTR,CTR_FIX_CANDIDATE,Higher confidence: strong search volume.,Could be wrong because the item has no observe...
4,5,content_44f34c0a90047651,REVIEW_CTR,CTR_FIX_CANDIDATE,Higher confidence: strong search volume.,Could be wrong because the item has no observe...
5,6,content_fec55986a1868d62,REVIEW_CTR,CTR_FIX_CANDIDATE,Higher confidence: strong search volume.,Could be wrong because the item has no observe...
6,7,content_44f34c0a90047651,REVIEW_CTR,CTR_FIX_CANDIDATE,Higher confidence: strong search volume.,Could be wrong if the observed CTR does not re...
7,8,content_44f34c0a90047651,REVIEW_CTR,CTR_FIX_CANDIDATE,Higher confidence: strong search volume.,Could be wrong if the observed CTR does not re...
8,9,content_9c057b66c30a3abb,REVIEW_CTR,CTR_FIX_CANDIDATE,Higher confidence: strong search volume.,Could be wrong because the item has no observe...
9,10,content_9c057b66c30a3abb,REVIEW_CTR,CTR_FIX_CANDIDATE,Higher confidence: strong search volume.,Could be wrong because the item has no observe...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Some picks may be weak when they have limited search evidence or when the observed CTR does not clearly indicate a useful improvement opportunity. These rows should be treated as review candidates rather than guaranteed recommendations.

Leakage check: The baseline uses only March 2026 Google Search Console fields available at the decision moment, including impressions and clicks. It does not use future-window outcomes, product flags, trend labels, or label-derived fields. Therefore, the baseline does not intentionally use future or label-derived information.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Question 4: Weak picks and leakage check

# Find potentially weak recommendations among the top 20
weak_picks = top20[
    (top20["gsc_impressions"] < 300) |
    (top20["gsc_clicks"] == 0)
][
    [
        "rank",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr_pct",
        "score",
        "reason_code",
        "action"
    ]
]

print("Potentially weak picks:")
weak_picks

# Leakage check: list the fields actually used by the baseline
baseline_inputs = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_data_available"
]

print("\nFields used by the baseline:")
for field in baseline_inputs:
    print("-", field)

print("\nLeakage check:")
print("✓ No future-window outcome fields were used.")
print("✓ No product flags were used.")
print("✓ No label-derived fields were used.")
print("✓ Only March 2026 observed GSC signals were used.")

Potentially weak picks:

Fields used by the baseline:
- report_date
- client_hash_id
- content_hash_id
- gsc_impressions
- gsc_clicks
- gsc_data_available

Leakage check:
✓ No future-window outcome fields were used.
✓ No product flags were used.
✓ No label-derived fields were used.
✓ Only March 2026 observed GSC signals were used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.